TimeSeventeen BTC 5m on-chain fill data (not order book / quote)

uses daily aligned dataset , slug rule - r"^btc-updown-5m-[0-9]+$" , 

In [2]:
from datetime import date, timedelta
from pathlib import Path

import polars as pl
from huggingface_hub import hf_hub_download

DATASET_NAME = "TimeSeventeen/Polymarket-v1"
START_DATE = date(2026, 2, 12) # 5m crypto up/down launch date (feb 12 2026)
END_DATE = date(2026, 4, 28)
PREDICTION_HORIZON_SECONDS = 60
BTC_5M_SLUG_PATTERN = r"^btc-updown-5m-[0-9]+$"

RAW_DATA_DIR = Path("../data/raw/timeseventeen")
PROCESSED_DATA_DIR = Path("../data/processed")
OUTPUT_PATH = PROCESSED_DATA_DIR / "timeseventeen_btc5m_v0.parquet"

# One lookback partition is required for markets closing just after midnight.
DOWNLOAD_START_DATE = START_DATE - timedelta(days=1)

c:\Users\Anigma PC\miniconda3\envs\btc_research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

partition_dates = []
current_date = DOWNLOAD_START_DATE
while current_date <= END_DATE:
    partition_dates.append(current_date)
    current_date += timedelta(days=1)

for partition_date in partition_dates:
    filename = f"{partition_date:%Y_%m_%d}.parquet"
    local_path = RAW_DATA_DIR / "daily_aligned" / filename

    if local_path.exists():
        continue

    hf_hub_download(
        repo_id=DATASET_NAME,
        filename=f"daily_aligned/{filename}",
        repo_type="dataset",
        local_dir=RAW_DATA_DIR,
    )

print(f"Required partitions available: {len(partition_dates)}")

Required partitions available: 77



scan for btc 5-minute fills, partitions the dataset into each day of trading activity (includes midnight overlap activity -> rolling 48 hour window achieves this)

In [4]:
source_columns = [
    "block_timestamp",
    "usdc_amount",
    "condition_id",
    "outcome_seq",
    "neg_risk",
    "outcome_label",
    "winning_outcome_label",
    "resolution_status",
    "taker_base_fee",
    "maker_base_fee",
    "opens_at",
    "close_at",
    "resolved_at",
    "market_slug",
    "p_event",
    "D",  # D -> event normalized aggressor direction (+1 -> taker buys, -1 taker sells) 
]

def partition_path(day):
    return RAW_DATA_DIR / "daily_aligned" / f"{day:%Y_%m_%d}.parquet"

missing_files = [partition_path(day) for day in partition_dates if not partition_path(day).exists()]
assert not missing_files, f"Missing {len(missing_files)} required partitions"

def scan_btc5m_for_day(market_date):
    # Include the preceding trade-date partition for the market closing at midnight.
    day_files = [
        partition_path(market_date - timedelta(days=1)),
        partition_path(market_date),
    ]
    return (
        pl.scan_parquet(day_files)
        .select(source_columns)
        .filter(pl.col("market_slug").str.contains(BTC_5M_SLUG_PATTERN))
    )


(note: slug suffix = 5m window start , opens_at = time contract itself opened (can precede event window by hours))


code below constructs market metadata/labels

In [5]:
def build_markets_for_day(trades, market_date):
    markets = (
        trades
        .group_by("condition_id")
        .agg(
            pl.col("market_slug").first(),
            pl.col("opens_at").first(),
            pl.col("close_at").first(),
            pl.col("resolved_at").drop_nulls().first(),
            pl.col("resolution_status").drop_nulls().first(),
            pl.col("winning_outcome_label").drop_nulls().first(),
            pl.col("taker_base_fee").drop_nulls().first(),
            pl.col("maker_base_fee").drop_nulls().first(),
            pl.col("outcome_label").filter(pl.col("outcome_seq") == 1).drop_nulls().unique().first().alias("reference_outcome"),
            pl.col("outcome_label").filter(pl.col("outcome_seq") == 2).drop_nulls().unique().first().alias("opposite_outcome"),
            pl.col("market_slug").n_unique().alias("slug_values"),
            pl.col("close_at").n_unique().alias("close_values"),
            pl.col("winning_outcome_label").drop_nulls().n_unique().alias("winner_values"),
        )
        .filter(pl.col("close_at").dt.date() == market_date)
        .with_columns(
            pl.col("opens_at").dt.epoch("s").alias("start_ts"),
            pl.col("close_at").dt.epoch("s").alias("end_ts"),
            pl.col("market_slug").str.extract(r"([0-9]+)$", 1).cast(pl.Int64).alias("window_start_ts"),
            pl.when(pl.col("winning_outcome_label") == "Up").then(pl.lit(1))
            .when(pl.col("winning_outcome_label") == "Down").then(pl.lit(0))
            .otherwise(None).cast(pl.Int8).alias("resolution"),
        )
        .with_columns((pl.col("end_ts") - PREDICTION_HORIZON_SECONDS).alias("prediction_ts"))
        .collect()
    )
    assert markets["condition_id"].n_unique() == markets.height
    assert markets.select(pl.max_horizontal("slug_values", "close_values", "winner_values").max()).item() == 1
    assert (markets["end_ts"] - markets["window_start_ts"] == 300).all()
    assert (markets["reference_outcome"] == "Up").all()
    assert (markets["opposite_outcome"] == "Down").all()
    return markets



prediction time price / order flow features 

all market data computed at/before prediction_ts

note: multiple on-chain fills can share a block timestamp and the cleaned table has no transaction-order field, the price at the latest eligible second is the USDC-weighted mean p_event across fills in that second.

In [6]:
## aggregate_flow -> calculates 

def aggregate_flow(trades, earliest_timestamp, prefix):
    return (
        trades
        .filter(pl.col("block_timestamp") >= earliest_timestamp)
        .group_by("condition_id")
        .agg(
            pl.len().alias(f"{prefix}_trade_count"),
            pl.col("usdc_amount").sum().alias(f"{prefix}_volume_usdc"),
            (pl.col("D") * pl.col("usdc_amount"))
            .sum()
            .alias(f"{prefix}_signed_flow_usdc"),
        )
        .with_columns(
            (
                pl.col(f"{prefix}_signed_flow_usdc")
                / pl.col(f"{prefix}_volume_usdc")
            ).alias(f"{prefix}_imbalance")
        )
    )

def build_features_for_day(trades, markets):
    market_times = markets.select(
        "condition_id", "window_start_ts", "prediction_ts", "end_ts"
    ).lazy()
    eligible = (
        trades.join(market_times, on="condition_id", how="inner")
        .filter(pl.col("block_timestamp") <= pl.col("prediction_ts"))
    )
    prices_by_second = (
        eligible.group_by(["condition_id", "block_timestamp"])
        .agg(
            ((pl.col("p_event") * pl.col("usdc_amount")).sum() / pl.col("usdc_amount").sum()).alias("up_price"),
            pl.len().alias("price_second_trade_count"),
            pl.col("usdc_amount").sum().alias("price_second_volume_usdc"),
        )
    )
    latest_price = (
        prices_by_second.group_by("condition_id")
        .agg(
            pl.col("block_timestamp").max().alias("price_timestamp"),
            pl.col("up_price").sort_by("block_timestamp").last(),
            pl.col("price_second_trade_count").sort_by("block_timestamp").last(),
            pl.col("price_second_volume_usdc").sort_by("block_timestamp").last(),
        )
    )
    window_flow = aggregate_flow(eligible, pl.col("window_start_ts"), "window")
    trailing_60s_flow = aggregate_flow(eligible, pl.col("prediction_ts") - 60, "trailing_60s")
    return latest_price, window_flow, trailing_60s_flow

constructing final feature matrix 

In [7]:
count_and_volume_columns = [
    "window_trade_count",
    "window_volume_usdc",
    "window_signed_flow_usdc",
    "trailing_60s_trade_count",
    "trailing_60s_volume_usdc",
    "trailing_60s_signed_flow_usdc",
]

def process_day(market_date):
    trades = scan_btc5m_for_day(market_date)
    markets = build_markets_for_day(trades, market_date)
    latest_price, window_flow, trailing_60s_flow = build_features_for_day(trades, markets)
    return (
        markets.lazy()
        .join(latest_price, on="condition_id", how="left")
        .join(window_flow, on="condition_id", how="left")
        .join(trailing_60s_flow, on="condition_id", how="left")
    .with_columns(
        pl.lit("BTC").alias("crypto"),
        pl.lit("5-minute").alias("timeframe"),
        (1.0 - pl.col("up_price")).alias("down_price"),
        (pl.col("prediction_ts") - pl.col("price_timestamp"))
        .alias("price_age_seconds"),
        (pl.col("end_ts") - pl.col("price_timestamp"))
        .alias("effective_seconds_to_close"),
        pl.col(count_and_volume_columns).fill_null(0),
    )
    .rename({"condition_id": "market_id"})
    .select(
        "market_id",
        "market_slug",
        "crypto",
        "timeframe",
        "resolution_status",
        "winning_outcome_label",
        "resolution",
        "opens_at",
        "close_at",
        "resolved_at",
        "start_ts",
        "end_ts",
        "window_start_ts",
        "prediction_ts",
        "price_timestamp",
        "price_age_seconds",
        "effective_seconds_to_close",
        "up_price",
        "down_price",
        "price_second_trade_count",
        "price_second_volume_usdc",
        "window_trade_count",
        "window_volume_usdc",
        "window_signed_flow_usdc",
        "window_imbalance",
        "trailing_60s_trade_count",
        "trailing_60s_volume_usdc",
        "trailing_60s_signed_flow_usdc",
        "trailing_60s_imbalance",
        "taker_base_fee",
        "maker_base_fee",
    )
        .sort("window_start_ts")
        .collect()
    )

daily_frames = []
market_date = START_DATE
while market_date <= END_DATE:
    daily_frame = process_day(market_date)
    daily_frames.append(daily_frame)
    print(f"{market_date}: {daily_frame.height} markets")
    market_date += timedelta(days=1)

df_final = pl.concat(daily_frames).sort("window_start_ts")
df_final.head()

2026-02-12: 279 markets
2026-02-13: 288 markets
2026-02-14: 288 markets
2026-02-15: 288 markets
2026-02-16: 288 markets
2026-02-17: 288 markets
2026-02-18: 288 markets
2026-02-19: 288 markets
2026-02-20: 288 markets
2026-02-21: 288 markets
2026-02-22: 288 markets
2026-02-23: 288 markets
2026-02-24: 288 markets
2026-02-25: 288 markets
2026-02-26: 288 markets
2026-02-27: 288 markets
2026-02-28: 288 markets
2026-03-01: 288 markets
2026-03-02: 288 markets
2026-03-03: 288 markets
2026-03-04: 288 markets
2026-03-05: 288 markets
2026-03-06: 288 markets
2026-03-07: 288 markets
2026-03-08: 288 markets
2026-03-09: 288 markets
2026-03-10: 288 markets
2026-03-11: 288 markets
2026-03-12: 288 markets
2026-03-13: 288 markets
2026-03-14: 288 markets
2026-03-15: 288 markets
2026-03-16: 288 markets
2026-03-17: 288 markets
2026-03-18: 288 markets
2026-03-19: 288 markets
2026-03-20: 288 markets
2026-03-21: 288 markets
2026-03-22: 288 markets
2026-03-23: 288 markets
2026-03-24: 288 markets
2026-03-25: 288 

market_id,market_slug,crypto,timeframe,resolution_status,winning_outcome_label,resolution,opens_at,close_at,resolved_at,start_ts,end_ts,window_start_ts,prediction_ts,price_timestamp,price_age_seconds,effective_seconds_to_close,up_price,down_price,price_second_trade_count,price_second_volume_usdc,window_trade_count,window_volume_usdc,window_signed_flow_usdc,window_imbalance,trailing_60s_trade_count,trailing_60s_volume_usdc,trailing_60s_signed_flow_usdc,trailing_60s_imbalance,taker_base_fee,maker_base_fee
str,str,str,str,str,str,i8,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",i64,i64,i64,i64,i64,i64,i64,f64,f64,u32,f64,u32,f64,f64,f64,u32,f64,f64,f64,f64,f64
"""0xcdef2b38464c8deee66068a057c2…","""btc-updown-5m-1770856800""","""BTC""","""5-minute""","""resolved""","""Up""",1,2026-02-12 00:41:35.711515 UTC,2026-02-12 00:45:00 UTC,2026-02-12 00:45:53 UTC,1770856895,1770857100,1770856800,1770857040,1770856929,111,171,0.51,0.49,1,0.98,1,0.98,0.98,1.0,0,0.0,0.0,null,1000.0,1000.0
"""0xf2e66a8994ed3b7260341303a577…","""btc-updown-5m-1770857100""","""BTC""","""5-minute""","""resolved""","""Down""",0,2026-02-12 00:41:37.147620 UTC,2026-02-12 00:50:00 UTC,2026-02-12 00:50:29 UTC,1770856897,1770857400,1770857100,1770857340,null,null,null,null,null,null,null,0,0.0,0.0,null,0,0.0,0.0,null,1000.0,1000.0
"""0xc23f9721c8a78e1122af6f278d8b…","""btc-updown-5m-1770857400""","""BTC""","""5-minute""","""resolved""","""Up""",1,2026-02-12 00:41:37.146124 UTC,2026-02-12 00:55:00 UTC,2026-02-12 00:55:29 UTC,1770856897,1770857700,1770857400,1770857640,null,null,null,null,null,null,null,0,0.0,0.0,null,0,0.0,0.0,null,1000.0,1000.0
"""0x899c7e11bfd9f3dbaad773e8d2db…","""btc-updown-5m-1770857700""","""BTC""","""5-minute""","""resolved""","""Down""",0,2026-02-12 00:41:35.657404 UTC,2026-02-12 01:00:00 UTC,2026-02-12 01:01:23 UTC,1770856895,1770858000,1770857700,1770857940,null,null,null,null,null,null,null,0,0.0,0.0,null,0,0.0,0.0,null,1000.0,1000.0
"""0xed696022a7f0c6172502ee4cb022…","""btc-updown-5m-1770858000""","""BTC""","""5-minute""","""resolved""","""Up""",1,2026-02-12 00:41:35.681394 UTC,2026-02-12 01:05:00 UTC,2026-02-12 01:05:31 UTC,1770856895,1770858300,1770858000,1770858240,1770858151,89,149,0.01,0.99,1,0.06,3,1.090101,0.949899,0.871386,0,0.0,0.0,null,1000.0,1000.0



temp diagnostics 

fully day (24h) = 288 5m markets

In [8]:
assert df_final["market_id"].n_unique() == df_final.height
assert df_final["resolution"].null_count() == 0
assert df_final.filter(
    pl.col("price_timestamp") > pl.col("prediction_ts")
).is_empty()
assert df_final.filter(
    pl.col("up_price").is_not_null()
    & ~pl.col("up_price").is_between(0, 1, closed="both")
).is_empty()

daily_market_counts = (
    df_final
    .with_columns(pl.col("close_at").dt.date().alias("market_date"))
    .group_by("market_date")
    .agg(
        pl.len().alias("markets"),
        pl.col("resolution").is_not_null().sum().alias("resolved"),
        pl.col("up_price").is_not_null().sum().alias("has_price"),
        (pl.col("price_age_seconds") <= 60).sum().alias("price_age_le_60"),
    )
    .sort("market_date")
)
market_gaps = (
    df_final
    .with_columns(pl.col("window_start_ts").diff().alias("gap_seconds"))
    .filter(pl.col("gap_seconds") > 300)
    .with_columns(((pl.col("gap_seconds") // 300) - 1).alias("missing_intervals"))
    .select("market_slug", "close_at", "gap_seconds", "missing_intervals")
)

print(f"BTC five-minute markets: {df_final.height:,}")
print(f"Resolved labels: {df_final['resolution'].is_not_null().sum():,}")
print(f"Markets with a causal price: {df_final['up_price'].is_not_null().sum():,}")
print(f"Markets with price age <= 60s: {(df_final['price_age_seconds'] <= 60).sum():,}")
print(f"Markets with window flow: {(df_final['window_trade_count'] > 0).sum():,}")
print(f"Missing five-minute intervals inside the observed range: {market_gaps['missing_intervals'].sum():,}")

display(
    daily_market_counts,
    daily_market_counts.filter(pl.col("markets") != 288),
    market_gaps,
    df_final["price_age_seconds"].describe(),
    df_final.group_by("resolution").len().sort("resolution"),
)

BTC five-minute markets: 21,751
Resolved labels: 21,751
Markets with a causal price: 21,744
Markets with price age <= 60s: 21,661
Markets with window flow: 21,686
Missing five-minute intervals inside the observed range: 95


market_date,markets,resolved,has_price,price_age_le_60
date,u32,u32,u32,u32
2026-02-12,279,279,275,273
2026-02-13,288,288,288,286
2026-02-14,288,288,288,288
2026-02-15,288,288,288,288
2026-02-16,288,288,288,288
…,…,…,…,…
2026-04-24,288,288,288,288
2026-04-25,288,288,288,288
2026-04-26,288,288,288,288


market_date,markets,resolved,has_price,price_age_le_60
date,u32,u32,u32,u32
2026-02-12,279,279,275,273
2026-04-15,286,286,286,286
2026-04-16,286,286,286,286
2026-04-28,164,164,164,133


market_slug,close_at,gap_seconds,missing_intervals
str,"datetime[μs, UTC]",i64,i64
"""btc-updown-5m-1776255300""",2026-04-15 12:20:00 UTC,900,2
"""btc-updown-5m-1776380400""",2026-04-16 23:05:00 UTC,600,1
"""btc-updown-5m-1776382800""",2026-04-16 23:45:00 UTC,600,1
"""btc-updown-5m-1777377000""",2026-04-28 11:55:00 UTC,1500,4
"""btc-updown-5m-1777380900""",2026-04-28 13:00:00 UTC,3900,12
…,…,…,…
"""btc-updown-5m-1777405500""",2026-04-28 19:50:00 UTC,600,1
"""btc-updown-5m-1777406100""",2026-04-28 20:00:00 UTC,600,1
"""btc-updown-5m-1777406700""",2026-04-28 20:10:00 UTC,600,1


statistic,value
str,f64
"""count""",21744.0
"""null_count""",7.0
"""mean""",96.928992
"""std""",2683.160629
"""min""",0.0
"""25%""",0.0
"""50%""",1.0
"""75%""",1.0
"""max""",85774.0


resolution,len
i8,u32
0,10758
1,10993


save df_final

In [9]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
df_final.write_parquet(OUTPUT_PATH)
%store df_final

print(f"Saved {df_final.height:,} markets to {OUTPUT_PATH}")

Stored 'df_final' (DataFrame)
Saved 21,751 markets to ..\data\processed\timeseventeen_btc5m_v0.parquet
